# Opdracht 2: Voorspel verhuuraantallen

**Teamnaam: Bakabana pindasaus**

Aroetin Avanesi, aroetin 25103865

Moreo Rahiembaks, moreorahiembaks, 25107607

Kaylim Taai, kaylimtaai, 24093157

## Inhoudsopgave
1. Inleiding
2. Exploratieve Data Analyse
3. Time series feature engineering
4. Modelleren
5. Bevindingen
6. Conclusie en aanbevelingen
7. Referenties

## 1. Inleiding

CityBike verhuurt fietsen op verschillende locaties in een grote stad. Klanten kunnen op ieder moment van de dag een fiets huren, bijvoorbeeld om naar hun werk te gaan, om de stad te bekijken of gewoon om te recreëren. Als er te weinig fietsen staan, kan een klant geen fiets huren en loopt CityBike omzet mis. Als er juist te veel fietsen op één plek staan, worden fietsen en stallingsplaatsen niet goed benut.

De data lijkt sterk op de bekende Bike Sharing dataset van Capital Bikeshare in Washington D.C. (Fanaee-T & Gama, 2014). De kolommen, de periode (2011 en 2012) en de feestdagen komen daarmee overeen. Dat helpt ons om de genormaliseerde weerkolommen terug te rekenen naar echte eenheden (zie 2.2) en om opvallende momenten in de data te verklaren.

### 1.1 Probleemstelling

Ons team krijgt de vraag om op basis van historische gegevens te voorspellen **hoeveel fietsen er per uur worden verhuurd** (`cnt`). We trainen op de periode van 1 januari 2011 tot en met 30 november 2012 en voorspellen de periode van **1 tot en met 19 december 2012**.

Dit is een regressieprobleem met een tijdreeks. Dat heeft drie gevolgen voor onze aanpak:

1. **De volgorde in de tijd is belangrijk.** We mogen bij het valideren nooit trainen op data die na de validatieperiode ligt. Daarom gebruiken we geen gewone KFold maar een tijdgebaseerde cross-validation (hoofdstuk 4.1).
2. **We voorspellen 19 dagen vooruit.** Voor de testperiode kennen we `cnt` niet. Een feature als "het aantal verhuringen van een uur geleden" kunnen we daar dus niet gebruiken. Daar houden we rekening mee bij de feature engineering (hoofdstuk 3.3).
3. **De score is de RMSE.** Kaggle beoordeelt ons op de Root Mean Squared Error. De minimumeis is een RMSE van 80.

Het doel is een model dat CityBike helpt om de fietsen beter te verdelen, zodat klanten zo vaak mogelijk een fiets kunnen huren wanneer ze die nodig hebben.

## 2. Exploratieve Data Analyse

### 2.1 Data inladen

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Vaste kleuren, zodat dezelfde groep in iedere grafiek dezelfde kleur heeft
BLUE = "#257de0"
ORANGE = "#f47230"
sns.set_theme(style="whitegrid")

TARGET = "cnt"
WEATHER_COLS = ["temp", "atemp", "hum", "windspeed"]


class BikeData:
    """Laadt de datasets van de Kaggle competitie Voorspel verhuuraantallen."""

    def __init__(self, data_dir="voorspel-verhuuraantallen-2026"):
        # Map waarin de csv bestanden staan (in de GitHub repo)
        self.data_dir = Path(data_dir)
        self.train = None
        self.test = None
        self.submission = None

    def load(self):
        """Leest de drie csv bestanden in en zet date_hour om naar datetime."""
        self.train = pd.read_csv(self.data_dir / "train.csv",
                                 parse_dates=["date_hour"])
        self.test = pd.read_csv(self.data_dir / "test.csv",
                                parse_dates=["date_hour"])
        self.submission = pd.read_csv(
            self.data_dir / "sample_submission.csv"
        )
        return self

    def check_shapes(self):
        """Toont het aantal rijen en kolommen en de periode per dataset."""
        for name, df in [("train", self.train), ("test", self.test)]:
            print(f"{name:<6} {df.shape}  "
                  f"{df['date_hour'].min()} t/m {df['date_hour'].max()}")
        print(f"submission {self.submission.shape}")


data = BikeData().load()
data.check_shapes()
data.train.head()

train  (16637, 8)  2011-01-01 00:00:00 t/m 2012-11-30 23:00:00
test   (456, 7)  2012-12-01 00:00:00 t/m 2012-12-19 23:00:00
submission (456, 2)


,date_hour,holiday,weathersit,temp,atemp,hum,windspeed,cnt
0,2011-01-01 00:00:00,0,1,0.24,0.2879,0.81,0.0,16
1,2011-01-01 01:00:00,0,1,0.22,0.2727,0.80,0.0,40
2,2011-01-01 02:00:00,0,1,0.22,0.2727,0.80,0.0,32
3,2011-01-01 03:00:00,0,1,0.24,0.2879,0.75,0.0,13
4,2011-01-01 04:00:00,0,1,0.24,0.2879,0.75,0.0,1


### 2.2 Uitleg kolommen

De weerkolommen zijn genormaliseerd naar een waarde tussen 0 en 1. Volgens de beschrijving van de originele dataset (Fanaee-T, 2013) kun je ze zo terugrekenen naar echte eenheden. Dat gebruiken we in de grafieken, omdat "20 °C" makkelijker te lezen is dan "0,6".

| Kolom | Betekenis | Type | Terugrekenen |
|---|---|---|---|
| `date_hour` | Datum en uur van de meting | datetime | |
| `holiday` | 1 = officiële feestdag in Washington D.C., 0 = geen feestdag | binair | |
| `weathersit` | Weertype: 1 = helder, 2 = mist of bewolkt, 3 = lichte regen of sneeuw, 4 = zware regen, hagel of sneeuw | categorisch (geordend) | |
| `temp` | Temperatuur | continu | $t = temp \cdot 47 - 8$ (°C) |
| `atemp` | Gevoelstemperatuur | continu | $t = atemp \cdot 66 - 16$ (°C) |
| `hum` | Luchtvochtigheid | continu | $hum \cdot 100$ (%) |
| `windspeed` | Windsnelheid | continu | $windspeed \cdot 67$ (km/u) |
| `cnt` | **Target:** aantal verhuurde fietsen in dat uur | geheel getal | |

`weathersit` is een getal, maar eigenlijk een categorie met een volgorde: hoe hoger, hoe slechter het weer. Een model mag daarom ook rekenen met "hoger = slechter".

### 2.3 Datatypes en basisstatistieken

### 2.4 Tijdserie-elementen en patronen

### 2.5 Relaties tussen de variabelen en cnt

### 2.6 Belangrijkste bevindingen

### 2.7 Data opschonen

## 3. Time series feature engineering

### 3.1 Seizoenspatronen (Fourier analyse)

### 3.2 Trends

### 3.3 Cycli en autocorrelatie

### 3.4 Andere tijdsgebaseerde features

### 3.5 Invloed van de features en uiteindelijke keuze

## 4. Modelleren

### 4.1 Aanpak: RMSE, cross-validation en hyperparameter tuning

_4.2 t/m 4.6: modellen met time series features + reguliere features (minimaal 5 modellen)._

### 4.2 Lineaire regressie

#### Uitleg
_(wiskundige uitleg, met o.a. loss function en regularisatie)_
#### Resultaten

### 4.3 Ensemble model

#### Uitleg
_(wiskundige uitleg)_
#### Resultaten

### 4.4 Model 3 (eigen keuze)

#### Uitleg
_(onderbouwing keuze)_
#### Resultaten

### 4.5 Model 4 (eigen keuze)

#### Uitleg
_(onderbouwing keuze)_
#### Resultaten

### 4.6 Model 5 (eigen keuze)

#### Uitleg
_(onderbouwing keuze)_
#### Resultaten

### 4.7 SARIMA(X)

#### Uitleg
#### Resultaten

### 4.8 Tweede tijdseriemodel (eigen keuze)

#### Uitleg
#### Resultaten

### 4.9 Hybride model

#### Uitleg
#### Resultaten

### 4.10 Voorspellingen voor Kaggle

## 5. Bevindingen

### 5.1 Samenvatting

### 5.2 Overzicht resultaten

| Model | Beste parameters | RMSE (CV) | RMSE Kaggle |
|---|---|---|---|
| Lineaire regressie | | | |
| Ensemble model | | | |
| Model 3 | | | |
| Model 4 | | | |
| Model 5 | | | |
| SARIMA(X) | | | |
| Tijdseriemodel 2 | | | |
| Hybride model | | | |

### 5.3 Welk model presteert het beste?

### 5.4 Kaggle-scores en verbeteringen

| Submission | Datum | Model / wijziging | RMSE Kaggle |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |

## 6. Conclusie en aanbevelingen

### 6.1 Welke features hebben de meeste invloed?

### 6.2 Aansluiting bij de EDA

### 6.3 Advies aan CityBike

### 6.4 Welk model adviseren wij en waarom?

## 7. Referenties